# Story Generation

This notebook loads a trained Transformer model and generates stories based on a prompt.

In [1]:
%load_ext autoreload
%autoreload 2

import torch
from tokenizers import Tokenizer
from model import Transformer, Sampler
from config import ModelConfig, TrainerConfig
from safetensors.torch import load_model
import torch.nn.functional as F

In [ ]:
# Setup device and load tokenizer
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
tokenizer = Tokenizer.from_file("tokenizer.json")
eot_token_id = tokenizer.token_to_id("<|endoftext|>")

# Initialize model with the same config used during training
config = ModelConfig(vocab_size=tokenizer.get_vocab_size())
model = Transformer(config).to(device)

# Load weights
try:
    load_model(model, "model.safetensors")
    print("Successfully loaded model.safetensors")
except FileNotFoundError:
    print("Warning: model.safetensors not found. Ensure you have trained the model first.")

model.eval()
sampler = Sampler(model, temperature=0.8)

In [8]:
@torch.no_grad()
def generate_story(prompt, max_tokens=50):
    # Encode prompt
    encoded = tokenizer.encode(prompt)
    idx = torch.tensor([encoded.ids], dtype=torch.long, device=device)
    
    print(f"Prompt: {prompt}\n" + "-" * 30)
    
    generated_ids = encoded.ids
    
    for _ in range(max_tokens):
        # Crop index to context length
        idx_cond = idx[:, -config.context_length:]
        
        # Get predictions
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / sampler.temperature
        
        # Sample next token
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        
        # Append to sequence
        idx = torch.cat((idx, idx_next), dim=1)
        token_id = idx_next.item()
        generated_ids.append(token_id)
        
        # Stop if EOT is reached
        if token_id == eot_token_id:
            break
            
    return tokenizer.decode(generated_ids)

In [9]:
prompt = "Once upon a time, there was a little cat named"
story = generate_story(prompt)
print(story)

Prompt: Once upon a time, there was a little cat named
------------------------------
Once upon a time, there was a little cat named Ducky. hall. One day, Timmy wanted to go to the park. He called a little bit very high, but he couldn't find it.

Desday, he found a big, shiny voice. He was very scared but he
